In [ ]:
# One-cell Colab: enable a T4 GPU, then run this cell.
import os, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

BRANCH = 'feature/hada-full-mamba'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/content/tcformer-full-mamba')

if Path('/content/drive/MyDrive').is_dir():
    DRIVE_ROOT = Path('/content/drive')
else:
    mountpoint = Path('/content/drive')
    if mountpoint.exists() and any(mountpoint.iterdir()):
        mountpoint = Path('/content/google_drive')
    drive.mount(str(mountpoint))
    DRIVE_ROOT = mountpoint

MNE_DATA = DRIVE_ROOT / 'MyDrive/datasets/PhysioNetMI_MNE'
RESULT_ARCHIVE = DRIVE_ROOT / 'MyDrive/TCFormer-results/full_mamba_physionet20_4p1s_bs96_ep125_results'

def run(command, cwd=None, env=None, stream=False):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if not stream:
        subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, check=True)
        return
    # Inherit Colab's stdout/stderr directly. This avoids PIPE buffering and
    # also preserves progress output that uses carriage returns instead of newlines.
    started_at = time.monotonic()
    process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, env=env)
    while True:
        try:
            return_code = process.wait(timeout=30)
            break
        except subprocess.TimeoutExpired:
            elapsed_minutes = (time.monotonic() - started_at) / 60
            print(f'[Colab heartbeat] training process is running | elapsed={elapsed_minutes:.1f}m', flush=True)
    if return_code != 0:
        raise RuntimeError(f'Command failed with exit code {return_code}')

run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    run(['git', 'checkout', BRANCH], cwd=REPO_PATH)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)

UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1', '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)
run([PYTHON, '-c', "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); assert torch.cuda.is_available(), 'Colab GPU is not enabled'; print('GPU:', torch.cuda.get_device_name(0))"])

override = "import yaml; from pathlib import Path; p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); c['max_epochs_loso']=125; c['preprocessing']['physionet']['trial_duration']=4.1; c['preprocessing']['physionet']['batch_size']=96; p.write_text(yaml.safe_dump(c, sort_keys=False))"
run([PYTHON, '-c', override], cwd=REPO_PATH)

MNE_DATA.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment.update({'PYTHONUNBUFFERED': '1', 'MPLBACKEND': 'Agg', 'MNE_DATA': str(MNE_DATA), 'MNE_DATASETS_EEGBCI_PATH': str(MNE_DATA)})
print('===== TRAIN FULL-MAMBA HADA-TCFORMER | PHYSIONET S001-S020 LOSO | 4.1 s | BS 96 | 125 EPOCHS =====', flush=True)
run([PYTHON, '-u', 'train_pipeline.py', '--model', 'hada_tcformer', '--dataset', 'physionet', '--loso', '--gpu_id', '0'], cwd=REPO_PATH, env=environment, stream=True)

RESULT_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Train complete. Results:', archive, flush=True)
